<a href="https://colab.research.google.com/github/aferenaz/MLOPSII-tp2/blob/main/mini_tp2_actividad_colab_Ferenaz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-TP 2 — Metadatos por GraphQL vs REST

**Andrea Ferenaz** · entrega Sesión 2 · *versión lista para Google Colab*

Expone los metadatos del **clasificador de tumores cerebrales (MRI)** por **GraphQL**
(Strawberry) y los compara con **REST**.
El notebook corre de punta a punta en Colab: los servidores se levantan en threads y para
ver **GraphiQL** en el navegador se usa el proxy de puertos de Colab.

## 0. Requisitos

En Colab usamos `pip` (no `uv`). Corré esta celda una sola vez.

In [ ]:
# Colab: instalar dependencias (uv no viene por defecto en Colab)
!pip install -q strawberry-graphql fastapi "uvicorn[standard]" requests

# Referencia de la consigna (entorno con uv):
# !uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 13.0 MB/s eta 0:00:00


## 1. Los metadatos de TU modelo

Modelo: **clasificador de tumores cerebrales sobre MRI** .
El esquema de la consigna usa métricas de clasificación (`auc`, `accuracy`, `f1`), que son
 las de este modelo.

In [ ]:
# TODO -> completado con tu modelo de Vision (FIUBA/CEIA). Cambiá por tus numeros reales del run.
MI_MODELO = {
    "name": "brain-tumor-classifier",    # clasificador de tumores cerebrales (MRI)
    "version": 1,                        # <- version
    "metrics": {                         # <- metricas reales de tu run
        "auc": 0.98,
        "accuracy": 0.95,
        "f1": 0.94,
    },
}

print("Modelo:", MI_MODELO["name"], "v" + str(MI_MODELO["version"]))
print("Métricas:", MI_MODELO["metrics"])

Modelo: brain-tumor-classifier v1
Métricas: {'auc': 0.98, 'accuracy': 0.95, 'f1': 0.94}


### (Opcional) Leer las métricas desde MLflow

Sin MLflow local.

In [ ]:
# Opcional: si lees de MLflow, descomentá y ajustá.
# Si algo falla, se queda con MI_MODELO["metrics"] y el notebook corre igual.
try:
    import mlflow
    from mlflow.tracking import MlflowClient
    # mlflow.set_tracking_uri("file:./mlruns")   # <- tu tracking uri
    client = MlflowClient()
    mv = client.get_latest_versions(MI_MODELO["name"])[0]   # última versión registrada
    run = client.get_run(mv.run_id)
    real = run.data.metrics
    for k in MI_MODELO["metrics"]:
        if k in real:
            MI_MODELO["metrics"][k] = round(float(real[k]), 4)
    MI_MODELO["version"] = int(mv.version)
    print("Métricas leídas de MLflow:", MI_MODELO["metrics"])
except Exception as e:
    print("Sin MLflow (uso los valores del dict). Detalle:", type(e).__name__)

Sin MLflow (uso los valores del dict). Detalle: ModuleNotFoundError


## 2. El esquema GraphQL

`Metrics` y `Model` quedan **tipados** en entrada y salida. El resolver `metrics` lee del dict

In [ ]:
import strawberry
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

@strawberry.type
class Metrics:
    auc: float
    accuracy: float
    f1: float

@strawberry.type
class Model:
    name: str
    version: int

    @strawberry.field
    def metrics(self) -> Metrics:
        m = MI_MODELO["metrics"]
        # Métricas reales (vienen del dict o de MLflow, ya resueltas arriba)
        return Metrics(auc=m["auc"], accuracy=m["accuracy"], f1=m["f1"])

@strawberry.type
class Query:
    @strawberry.field
    def model(self) -> Model:
        return Model(name=MI_MODELO["name"], version=MI_MODELO["version"])

schema = strawberry.Schema(query=Query)
app = FastAPI(title="Mini-TP 2 - metadatos por GraphQL")
app.include_router(GraphQLRouter(schema), prefix="/graphql")   # sirve GraphiQL por GET
print("Esquema GraphQL listo")

Esquema GraphQL listo


## 3. Levantar la API (en segundo plano) y consultar

En Colab el servidor va en un thread con su **propio event loop** (por eso el
`asyncio.new_event_loop()`).

In [ ]:
import threading, time, asyncio, uvicorn

def _run(app, port):
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")).run()

threading.Thread(target=_run, args=(app, 8010), daemon=True).start()
time.sleep(3)   # dar tiempo a que levante
print("GraphQL corriendo en http://127.0.0.1:8010/graphql")

GraphQL corriendo en http://127.0.0.1:8010/graphql


In [ ]:
import requests

# La query pide SOLO lo necesario (nada de sobre-traer)
query = "{ model { name version metrics { auc accuracy f1 } } }"
r = requests.post("http://127.0.0.1:8010/graphql", json={"query": query})
print(r.status_code)
import json; print(json.dumps(r.json(), indent=2, ensure_ascii=False))

200
{
  "data": {
    "model": {
      "name": "brain-tumor-classifier",
      "version": 1,
      "metrics": {
        "auc": 0.98,
        "accuracy": 0.95,
        "f1": 0.94
      }
    }
  }
}


### GraphiQL en el navegador (Colab)

En Colab `localhost` no se abre directo: hay que pasar por el **proxy de puertos**.

In [ ]:
# GraphiQL desde Colab vía proxy de puertos
try:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8010)")
    print("Abrí GraphiQL en:", url.rstrip("/") + "/graphql")
except Exception:
    # Fuera de Colab, es directo:
    print("Abrí GraphiQL en: http://127.0.0.1:8010/graphql")

# Query para pegar en GraphiQL:
print("""
{
  model {
    name
    version
    metrics { auc accuracy f1 }
  }
}
""")

Abrí GraphiQL en: https://8010-m-s-kkb-usc1c1-y66lov1enxo4-c.us-central1-1.prod.colab.dev/graphql

{
  model {
    name
    version
    metrics { auc accuracy f1 }
  }
}



### Cliente Python (probado arriba)

La llamada de la celda anterior ya es el cliente Python: un `POST` con el body `{"query": ...}`.
GraphQL usa **un solo endpoint** (`/graphql`) y **un solo verbo** (`POST`) para todo.

## 4. Comparación con REST

Levantamos el **mismo modelo** por REST y medimos cuántas **llamadas** y cuántos **bytes**
hacen falta para armar la misma vista. Diseño típico de REST: los metadatos y las métricas
viven en un recurso "gordo" (o repartidos en varios endpoints)

In [ ]:
# API REST equivalente (podés reusar la de la Sesión 1). Sirve el mismo modelo.
from fastapi import FastAPI as _FastAPI

rest = _FastAPI(title="Mini-TP 2 - REST")

@rest.get("/models/{name}")
def get_model(name: str):
    # REST devuelve el objeto completo (no podés pedir solo 'auc')
    return {
        "name": MI_MODELO["name"],
        "version": MI_MODELO["version"],
        "framework": "tensorflow-keras",
        "created_at": "2025-08-01T10:00:00",
        "description": "Clasificador de tumores cerebrales sobre MRI (glioma/meningioma/pituitary/none)",
        "stage": "Staging",
        "tags": {"team": "fiuba-ceia", "task": "image-classification"},
        "params": {"backbone": "EfficientNetB0", "epochs": 30, "batch_size": 32, "lr": 0.001},
        "metrics": {**MI_MODELO["metrics"], "precision": 0.95, "recall": 0.93, "loss": 0.18},
    }

@rest.get("/models/{name}/lineage")
def get_lineage(name: str):
    # El linaje es OTRO endpoint -> otra llamada en REST
    return {"inputs": ["mri_dataset", "augmentations", "train_val_split"], "run_id": "abc123"}

threading.Thread(target=_run, args=(rest, 8011), daemon=True).start()
time.sleep(3)
print("REST corriendo en http://127.0.0.1:8011")

REST corriendo en http://127.0.0.1:8011


In [ ]:
# --- Medición: armar la vista {name, version, auc} con cada enfoque ---

# GraphQL: 1 llamada, pidiendo SOLO lo necesario
gql_q = "{ model { name version metrics { auc } } }"
g = requests.post("http://127.0.0.1:8010/graphql", json={"query": gql_q})
gql_calls = 1
gql_bytes = len(g.content)

# REST: 1 llamada pero trae TODO el objeto (sobre-fetch)
rr = requests.get("http://127.0.0.1:8011/models/" + MI_MODELO["name"])
rest_calls = 1
rest_bytes = len(rr.content)

# Vista "modelo + linaje": GraphQL sigue en 1 llamada; REST necesita 2
rr2 = requests.get("http://127.0.0.1:8011/models/" + MI_MODELO["name"] + "/lineage")
rest_calls_lin = 2
rest_bytes_lin = len(rr.content) + len(rr2.content)

print(f"{'Vista':<28}{'GraphQL':>18}{'REST':>18}")
print("-" * 64)
print(f"{'name+version+auc (llam.)':<28}{gql_calls:>18}{rest_calls:>18}")
print(f"{'name+version+auc (bytes)':<28}{gql_bytes:>18}{rest_bytes:>18}")
print(f"{'+ linaje (llam.)':<28}{1:>18}{rest_calls_lin:>18}")
print(f"{'+ linaje (bytes)':<28}{gql_bytes:>18}{rest_bytes_lin:>18}")
print("-" * 64)
print(f"GraphQL trae {gql_bytes} B; REST trae {rest_bytes} B "
      f"({rest_bytes/gql_bytes:.1f}x mas) para la MISMA vista.")

Vista                                  GraphQL              REST
----------------------------------------------------------------
name+version+auc (llam.)                     1                 1
name+version+auc (bytes)                    87               454
+ linaje (llam.)                             1                 2
+ linaje (bytes)                            87               532
----------------------------------------------------------------
GraphQL trae 87 B; REST trae 454 B (5.2x mas) para la MISMA vista.


### Conclusión REST vs GraphQL

Para armar la vista `{name, version, auc}` del `brain-tumor-classifier`:

| | GraphQL | REST |
|---|---|---|
| Llamadas | 1 | 1 |
| Bytes | 87 | 454 |
| + linaje (llamadas) | 1 | 2 |
| + linaje (bytes) | 87 | 532 |

Para la misma lectura, GraphQL trajo **87 B en 1 llamada** y REST **454 B (5.2x más)**, porque REST devuelve el objeto completo (framework, params, tags, todas las métricas) aunque yo solo pida `auc`. Apenas la vista crece —sumar el linaje— GraphQL sigue en **1 llamada** mientras REST necesita **2 endpoints** (`/models/{name}` + `/models/{name}/lineage`). GraphQL evita el sobre-fetch y los round-trips extra pidiendo exactamente los campos que necesito; REST gana en simplicidad y cache HTTP cuando la vista es fija.

## 5. (Opcional +) Linaje desde Neo4j

Resolver que devuelve el linaje leyendo de Neo4j. Queda **desactivado por defecto**
(`USAR_NEO4J = False`) para no romper el run si no tenés instancia. Poné tus credenciales y
`True` para probarlo. Base: `graphql_neo4j_lineage.ipynb`.

In [ ]:
# Opcional: instalar el driver si vas a usar Neo4j
# !pip install -q neo4j

USAR_NEO4J = False   # <- ponelo en True cuando tengas una instancia

NEO4J_URI  = "bolt://localhost:7687"   # o tu Aura: neo4j+s://xxxx.databases.neo4j.io
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"

import strawberry
from typing import List

@strawberry.type
class LineageNode:
    name: str
    kind: str

def leer_linaje(model_name: str):
    if not USAR_NEO4J:
        # Fallback simulado para que la celda corra sin Neo4j
        return [LineageNode(name="mri_dataset", kind="dataset"),
                LineageNode(name="augmentations", kind="transform"),
                LineageNode(name=model_name, kind="model")]
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    q = """
    MATCH (m:Model {name:$name})<-[:FEEDS*1..]-(d)
    RETURN d.name AS name, labels(d)[0] AS kind
    """
    with driver.session() as s:
        rows = s.run(q, name=model_name).data()
    driver.close()
    return [LineageNode(name=r["name"], kind=r["kind"]) for r in rows]

@strawberry.type
class ModelConLinaje:
    name: str
    version: int
    @strawberry.field
    def metrics(self) -> Metrics:
        m = MI_MODELO["metrics"]
        return Metrics(auc=m["auc"], accuracy=m["accuracy"], f1=m["f1"])
    @strawberry.field
    def lineage(self) -> List[LineageNode]:
        return leer_linaje(self.name)

@strawberry.type
class QueryLin:
    @strawberry.field
    def model(self) -> ModelConLinaje:
        return ModelConLinaje(name=MI_MODELO["name"], version=MI_MODELO["version"])

schema_lin = strawberry.Schema(query=QueryLin)
res = schema_lin.execute_sync("{ model { name lineage { name kind } } }")
print("errores:", res.errors)
print("data:", res.data)

errores: None
data: {'model': {'name': 'brain-tumor-classifier', 'lineage': [{'name': 'mri_dataset', 'kind': 'dataset'}, {'name': 'augmentations', 'kind': 'transform'}, {'name': 'brain-tumor-classifier', 'kind': 'model'}]}}


##  checklist

- [x] Notebook corriendo de punta a punta.
- [x] Query GraphQL y su salida (paso 3).
- [x] Comparación REST vs GraphQL con números + conclusión (paso 4).
- [x] En el README del repo, una línea sobre la diferencia que notaste.

**Línea README:**

> Con GraphQL leí `{name, version, auc}` del modelo en 1 llamada trayendo solo esos campos (87 B); el mismo dato por REST me obligó a descargar el objeto completo (454 B, 5.2x más) y a sumar un endpoint extra para el linaje.